# Agentic BM25 with few-shot examples (WANDS demo)

ELI5 version: we build a simple keyword search tool (BM25), then let a GPT-5-mini agent use it. We also show the agent a few example judgments so it understands what a good result looks like.

This mirrors `configs/cheat-at-search/agentic_ecom_bm25_fewshot_gpt5_mini.yml` with these key settings:
- model: gpt-5-mini
- search_tools: [bm25]
- few_shot: 6 sample judgments

We use the WANDS dataset as a compact ecommerce-style dataset for teaching.

In [1]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@ee2526eb8bfac087dc3f90522cc7191032e47dfd
from cheat_at_search.data_dir import mount
try:
    mount(use_gdrive=True)
except ImportError:
    from pathlib import Path
    manual_path = str(Path.home() / ".search-experiments" / "cheat-at-search")
    mount(use_gdrive=False, manual_path=manual_path)

  Cloning https://github.com/softwaredoug/cheat-at-search.git (to revision ee2526eb8bfac087dc3f90522cc7191032e47dfd) to /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-_apsp5w9
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-_apsp5w9
  Running command git rev-parse -q --verify 'sha^ee2526eb8bfac087dc3f90522cc7191032e47dfd'
  Running command git fetch -q https://github.com/softwaredoug/cheat-at-search.git ee2526eb8bfac087dc3f90522cc7191032e47dfd
  Running command git checkout -q ee2526eb8bfac087dc3f90522cc7191032e47dfd
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit ee2526eb8bfac087dc3f90522cc7191032e47dfd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To 

## Get an OpenAI Key + load corpus

This will prompt you for an OpenAI Key to interact with GPT-5.

In [2]:
import logging
import numpy as np
import pandas as pd

from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("agentic_bm25_fewshot")

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].astype(str).to_numpy()
doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_id_lookup)}

corpus[["doc_id", "title", "description"]].head(3)

,doc_id,title,description
0,0,solid wood platform bed,"good , deep sleep can be quite difficult to ha..."
1,1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend..."
2,2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...


## Pick a small set of queries

ELI5: we only test a few questions so the notebook runs fast.

In [ ]:
QUERY_COUNT = 10
queries = judgments[["query", "query_id"]].drop_duplicates()
queries = queries.sample(n=QUERY_COUNT, random_state=7).reset_index(drop=True)
queries

## Build few-shot examples (6 rows)

Few-shot examples are short labeled snippets shown to the model. They teach it what good and bad results look like. This notebook builds 6 examples from judgments.

In [ ]:
import random

def _grade_column(judgments_df):
    for col in ("grade", "relevance", "rel", "label", "score"):
        if col in judgments_df.columns:
            return col
    return None

def _grade_to_emoji(grade, grade_levels):
    if not grade_levels:
        return "😐"
    if len(grade_levels) == 1:
        return "😐"
    if grade == grade_levels[0]:
        return "😭"
    if grade == grade_levels[-1]:
        return "😃"
    return "😐"

def _sorted_grades(values):
    def _coerce(value):
        try:
            return float(value)
        except (TypeError, ValueError):
            return None
    numeric = [value for value in values if _coerce(value) is not None]
    if len(numeric) == len(values):
        return sorted(numeric)
    return sorted(values, key=lambda value: str(value))

def build_few_shot_block(corpus_df, judgments_df, num_rows=6, seed=42):
    grade_col = _grade_column(judgments_df)
    if grade_col is None:
        raise ValueError("Judgments need a grade column.")
    pool = judgments_df.dropna(subset=[grade_col, "query", "doc_id"])
    grades = _sorted_grades(list(pool[grade_col].dropna().unique()))
    if not grades:
        return ""
    rng = random.Random(seed)
    grouped = {
        grade: pool[pool[grade_col] == grade].sample(
            frac=1.0, random_state=rng.randrange(1 << 30)
        )
        for grade in grades
    }
    queues = {grade: grouped[grade].iterrows() for grade in grades}
    samples = []
    while len(samples) < num_rows:
        advanced = False
        for grade in grades:
            try:
                _, row = next(queues[grade])
            except StopIteration:
                continue
            samples.append(row)
            advanced = True
            if len(samples) >= num_rows:
                break
        if not advanced:
            break

    corpus_lookup = None
    if "doc_id" in corpus_df.columns:
        corpus_lookup = corpus_df.set_index("doc_id", drop=False)

    lines = ["Few-shot examples (query, product, relevance):"]
    for row in samples:
        query = row.get("query")
        doc_id = row.get("doc_id")
        grade = row.get(grade_col)
        emoji = _grade_to_emoji(grade, grades)
        title = ""
        description = ""
        if corpus_lookup is not None and doc_id in corpus_lookup.index:
            match = corpus_lookup.loc[doc_id]
            if hasattr(match, "ndim") and match.ndim > 1:
                match = match.iloc[0]
            if hasattr(match, "get"):
                title = match.get("title", "")
                description = match.get("description", "")
        lines.extend(
            [
                f"Query: {query}",
                f"Doc ID: {doc_id}",
                f"Title: {title}",
                f"Description: {description}",
                f"Relevance: {emoji}",
                "",
            ]
        )
    return "
".join(lines).strip()

few_shot_block = build_few_shot_block(corpus, judgments, num_rows=6, seed=42)
few_shot_block

## Build the BM25 tool (exact signature)

We use the same tool shape as the repo. The agent can call this function to retrieve candidates.

In [ ]:
from typing import Union

from cheat_at_search.tokenizers import snowball_tokenizer
from searcharray.similarity import bm25_similarity

TITLE_BOOST = 10.0
DESCRIPTION_BOOST = 1.0
K1 = None
B = None

def search_bm25(
    keywords: str,
    top_k: int = 5,
    agent_state=None,
) -> list[dict[str, Union[str, int, float]]]:
    """Search a corpus using BM25 over title/description fields.

    Args:
        keywords: The search query string.
        top_k: The number of top results to return (max 100).

    Returns:
        Search results as a list of dictionaries with 'id', 'title',
        'description', and 'score' keys.
    """
    if top_k > 100:
        raise ValueError("top_k must be <= 100")
    if agent_state is not None:
        past_queries = agent_state.setdefault("past_queries", set())
        if keywords in past_queries:
            return []
        past_queries.add(keywords)

    bm25_scores = np.zeros(len(corpus))
    similarity = None
    if K1 is not None or B is not None:
        similarity = bm25_similarity(k1=K1 or 1.2, b=B or 0.75)
    for term in snowball_tokenizer(keywords):
        if similarity is None:
            bm25_scores += corpus["title_snowball"].array.score(term) * TITLE_BOOST
            bm25_scores += corpus["description_snowball"].array.score(term) * DESCRIPTION_BOOST
        else:
            bm25_scores += (
                corpus["title_snowball"].array.score(term, similarity=similarity)
                * TITLE_BOOST
            )
            bm25_scores += (
                corpus["description_snowball"].array.score(term, similarity=similarity)
                * DESCRIPTION_BOOST
            )

    top_k_indices = np.argsort(bm25_scores)[-top_k:][::-1]
    bm25_scores = bm25_scores[top_k_indices]
    top_rows = corpus.iloc[top_k_indices].copy()
    top_rows.loc[:, "score"] = bm25_scores

    results = []
    for _, row in top_rows.iterrows():
        result = {
            "id": row.get("doc_id", row.name),
            "title": row.get("title", ""),
            "description": row.get("description", ""),
            "score": row.get("score", 0.0),
        }
        if "path" in top_rows.columns:
            result["path"] = row.get("path", "")
        results.append(result)
    return results

search_bm25("salon chair", top_k=3)

## Build the agentic strategy

ELI5: the agent reads the system prompt, sees the few-shot examples, then calls BM25 to fetch candidates. It returns a ranked list of doc_ids.

In [ ]:
from pydantic import BaseModel, Field
from cheat_at_search.agent.openai_agent import OpenAIAgent
from cheat_at_search.strategy.strategy import SearchStrategy

SYSTEM_PROMPT_BASE = (
    "You take user search queries and use a search tool to find products.\n\n"
    "Look at the search tools you have, their limitations, how they work, etc when forming your plan.\n\n"
    "Finally return results to the user per the SearchResults schema, ranked best to worst.\n\n"
    "Gather results until you have 10 best matches you can find. It's important to return at least 10.\n\n"
    "It's very important you consider carefully the correct ranking as you'll be evaluated on\n"
    "how close that is to the average shoppers ideal ranking."
)

SYSTEM_PROMPT = SYSTEM_PROMPT_BASE
if few_shot_block:
    SYSTEM_PROMPT = SYSTEM_PROMPT.rstrip() + "\n\n" + few_shot_block + "\n"

class SearchResults(BaseModel):
    """The state of the search agent, which can be used to inform future reasoning and tool use."""
    ranked_results: list[str] = Field(
        description="Top ranked search results (their doc_ids) when complete"
    )

class AgenticBM25FewShot(SearchStrategy):
    def __init__(self, corpus, top_k=10, workers=1):
        super().__init__(corpus, top_k=top_k, workers=workers)
        self.agent = OpenAIAgent(
            tools=[search_bm25],
            model="openai/gpt-5-mini",
            response_model=SearchResults,
            reasoning_level="low",
        )

    def search(self, query, k=10):
        inputs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Find me: {query}"},
        ]
        agent_state = {"past_queries": set()}
        resp, inputs, _ = self.agent.chat(inputs=inputs, agent_state=agent_state, logger=logger)
        ranked = []
        if resp and resp.output_parsed:
            ranked = resp.output_parsed.ranked_results or []
        ranked = [str(doc_id) for doc_id in ranked]

        if len(ranked) < k:
            backfill = search_bm25(query, top_k=20, agent_state=agent_state)
            for row in backfill:
                doc_id = str(row.get("id"))
                if doc_id not in ranked:
                    ranked.append(doc_id)
                if len(ranked) >= k:
                    break

        ranked = ranked[:k]
        indices = [doc_id_to_index[doc_id] for doc_id in ranked if doc_id in doc_id_to_index]
        return indices, [1.0] * len(indices)

agentic = AgenticBM25FewShot(corpus)
agentic.search("salon chair", k=3)

## Run the strategy on a small batch

ELI5: we let the agent answer a few queries and record how good the rankings are.

In [ ]:
from cheat_at_search.search import run_strategy

graded = run_strategy(
    agentic,
    judgments,
    queries=queries["query"].tolist(),
    seed=7,
    show_progress=True,
    cache=False,
)

graded.head()

## Summarize metrics

We compute NDCG@10 and MRR to get a quick quality summary.

In [ ]:
from cheat_at_search.search import ndcgs, mrrs

ndcg_series = ndcgs(graded)
mrr_series = mrrs(graded)

pd.DataFrame(
    {
        "metric": ["NDCG@10", "MRR"],
        "value": [ndcg_series.mean(), mrr_series.mean()],
    }
)